In [1]:
import json
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, cohen_kappa_score
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack
import warnings
from gensim.models import Word2Vec
warnings.filterwarnings('ignore')

print("SVM Fake News Detector - Simple Model")
print("="*60)

SVM Fake News Detector - Simple Model


In [ ]:
DATA_PATH = "../data/raw/multinli_1.0/multinli_1.0_train.jsonl"

data = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        if item["gold_label"] != "-":
            data.append(item)

data = data

df = pd.DataFrame(data)[["sentence1", "sentence2", "gold_label"]]

print(f"Dataset: {len(df)} samples")
print("\nLabel distribution:")
print(df["gold_label"].value_counts())

Dataset: 392702 samples

Label distribution:
gold_label
contradiction    130903
neutral          130900
entailment       130899
Name: count, dtype: int64


In [3]:
label_mapping = {"entailment": 0, "neutral": 1, "contradiction": 2}

sentence1_texts = df["sentence1"].values
sentence2_texts = df["sentence2"].values
labels = df["gold_label"].map(label_mapping).values

print(f"Sentence1 shape: {sentence1_texts.shape}")
print(f"Sentence2 shape: {sentence2_texts.shape}")
print(f"Labels shape: {labels.shape}")
print(f"\nExample sentence1:\n{sentence1_texts[0][:100]}...")
print(f"Example sentence2:\n{sentence2_texts[0][:100]}...")

Sentence1 shape: (392702,)
Sentence2 shape: (392702,)
Labels shape: (392702,)

Example sentence1:
Conceptually cream skimming has two basic dimensions - product and geography....
Example sentence2:
Product and geography are what make cream skimming work. ...


In [4]:
print("Building feature matrix...")

# Separate word-level TF-IDF for each sentence
word_vectorizer = TfidfVectorizer(
    max_features=20000,
    max_df=0.9,
    min_df=2,
    ngram_range=(1, 2),
    sublinear_tf=True,
    norm='l2',
    analyzer='word'
)

X1_word = word_vectorizer.fit_transform(sentence1_texts)
X2_word = word_vectorizer.transform(sentence2_texts)

# Interaction features
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

print("Computing interaction features...")
# Word-level cosine similarity
cosine_sim_word = np.array([
    cosine_similarity(X1_word[i], X2_word[i])[0][0] 
    for i in range(X1_word.shape[0])
]).reshape(-1, 1)

# Element-wise operations on word vectors
X_diff = X1_word - X2_word
X_mult = X1_word.multiply(X2_word)

# Enhanced basic features (also as dictionary for evaluation)
def enhanced_features(s1_array, s2_array):
    """Returns each feature type separately as dictionary"""
    
    jaccard_list = []
    len1_list = []
    len2_list = []
    len_ratio_list = []
    len_diff_list = []
    neg1_list = []
    neg2_list = []
    neg_diff_list = []
    
    for s1, s2 in zip(s1_array, s2_array):
        words1 = s1.lower().split()
        words2 = s2.lower().split()
        set1, set2 = set(words1), set(words2)
        
        # Jaccard similarity
        jaccard_list.append(len(set1 & set2) / len(set1 | set2) if len(set1 | set2) > 0 else 0)
        
        # Length features
        len1, len2 = len(words1), len(words2)
        len1_list.append(len1)
        len2_list.append(len2)
        len_ratio_list.append(min(len1, len2) / max(len1, len2) if max(len1, len2) > 0 else 0)
        len_diff_list.append(abs(len1 - len2))
        
        # Negation words
        negations = {'not', 'no', 'never', 'neither', 'none', 'nobody', "n't", 'nothing', 'nowhere'}
        neg1 = sum(1 for w in words1 if w in negations)
        neg2 = sum(1 for w in words2 if w in negations)
        neg1_list.append(neg1)
        neg2_list.append(neg2)
        neg_diff_list.append(abs(neg1 - neg2))
    
    return {
        'jaccard': np.array(jaccard_list).reshape(-1, 1),
        'len1': np.array(len1_list).reshape(-1, 1),
        'len2': np.array(len2_list).reshape(-1, 1),
        'len_ratio': np.array(len_ratio_list).reshape(-1, 1),
        'len_diff': np.array(len_diff_list).reshape(-1, 1),
        'neg1': np.array(neg1_list).reshape(-1, 1),
        'neg2': np.array(neg2_list).reshape(-1, 1),
        'neg_diff': np.array(neg_diff_list).reshape(-1, 1)
    }

print("Extracting enhanced features...")
enhanced_feats = enhanced_features(sentence1_texts, sentence2_texts)

# Normalize numerical features
scaler = StandardScaler()

# Scale each feature separately so you can evaluate them individually
enhanced_feats_scaled = {k: scaler.fit_transform(v) for k, v in enhanced_feats.items()}

# Stack all features - now you can comment out individual features to test them!
X = hstack([
    X1_word,                    # Sentence1 word TF-IDF
    X2_word,                    # Sentence2 word TF-IDF
    X_diff,                     # Difference (word-level)
    X_mult,                     # Multiplication (word-level)
    cosine_sim_word,            # Word cosine similarity
    #enhanced_feats_scaled['jaccard'],
    #enhanced_feats_scaled['len1'],
    enhanced_feats_scaled['len2'],
    enhanced_feats_scaled['len_ratio'],
    #enhanced_feats_scaled['len_diff'],
    enhanced_feats_scaled['neg1'],
    #enhanced_feats_scaled['neg2'],
    enhanced_feats_scaled['neg_diff']
])

print(f"\n✓ Final feature shape: {X.shape}")
print(f"  - Word TF-IDF (both sentences): {X1_word.shape[1] * 2}")
print(f"  - Interactions (diff + mult + cosine): {X1_word.shape[1] * 2 + 1}")
print(f"  - Enhanced handcrafted: {len(enhanced_feats_scaled)}")


Building feature matrix...
Computing interaction features...
Extracting enhanced features...

✓ Final feature shape: (392702, 80005)
  - Word TF-IDF (both sentences): 40000
  - Interactions (diff + mult + cosine): 40001
  - Enhanced handcrafted: 8


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42, stratify=labels
)


# Scale features only for the SVM branch (keeps sparsity)
from sklearn.preprocessing import MaxAbsScaler


svm_scaler = MaxAbsScaler()
X_train_svm = svm_scaler.fit_transform(X_train)
X_test_svm = svm_scaler.transform(X_test)


print(f"Train samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")
print(f"SVM-scaled train matrix shape: {X_train_svm.shape}")

Train samples: 314161
Test samples: 78541
Features: 80005
SVM-scaled train matrix shape: (314161, 80005)


In [6]:
print("Training Random Forest")

rf_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=60,
    min_samples_split=20,
    min_samples_leaf=6,
    max_features='sqrt',
    class_weight='balanced',
    bootstrap=True,
    random_state=43,
    n_jobs=-1,
    verbose=0
)

rf_model.fit(X_train, y_train)
print("✓ Training complete")

Training Random Forest
✓ Training complete


In [7]:
y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Cohen's Kappa: {kappa:.4f}")

print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
target_names = ["Entailment (VRAI)", "Neutral (À_VÉRIFIER)", "Contradiction (FAUX)"]
print(classification_report(y_test, y_pred, target_names=target_names))

Test Accuracy: 0.5748
Cohen's Kappa: 0.3622

CLASSIFICATION REPORT
                      precision    recall  f1-score   support

   Entailment (VRAI)       0.54      0.67      0.60     26180
Neutral (À_VÉRIFIER)       0.55      0.58      0.57     26180
Contradiction (FAUX)       0.66      0.47      0.55     26181

            accuracy                           0.57     78541
           macro avg       0.59      0.57      0.57     78541
        weighted avg       0.59      0.57      0.57     78541



In [8]:
print("Evaluating need for dimensionality reduction (default: off for max accuracy)...")


use_svd = False  # keep False to retain all signal; set True only if memory is tight


if use_svd:
    print("Applying TruncatedSVD (PCA for sparse matrices)...")
    print(f"Original feature dimension (scaled for SVM): {X_train_svm.shape[1]}")
    n_components = 500
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    X_train_reduced = svd.fit_transform(X_train_svm)
    X_test_reduced = svd.transform(X_test_svm)
    explained_variance = svd.explained_variance_ratio_.sum()
    print(f"Reduced feature dimension: {X_train_reduced.shape[1]}")
    print(f"Explained variance: {explained_variance:.4f} ({explained_variance*100:.2f}%)")
    X_train_use, X_test_use = X_train_reduced, X_test_reduced
else:
    print("Skipping SVD; using scaled full sparse feature space for the classifier.")
    X_train_use, X_test_use = X_train_svm, X_test_svm


print("✓ Feature matrix ready for SVM branch")

Evaluating need for dimensionality reduction (default: off for max accuracy)...
Skipping SVD; using scaled full sparse feature space for the classifier.
✓ Feature matrix ready for SVM branch


In [9]:
print("Training LinearSVC with optimized parameters on selected features...")


# LinearSVC on full sparse features (dual formulation disabled since n_samples > n_features)
svm_model = LinearSVC(
    C=1.5,                    # slightly stronger fitting to capture nuances
    max_iter=5000,            # allow more iterations for convergence
    random_state=42,
    dual=False,               # prefer primal when samples exceed features
    class_weight='balanced',
    loss='squared_hinge'
 )


svm_model.fit(X_train_use, y_train)
print("✓ Training complete")

Training LinearSVC with optimized parameters on selected features...
✓ Training complete


In [10]:
y_pred = svm_model.predict(X_test_use)
accuracy = accuracy_score(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)


print(f"Test Accuracy: {accuracy:.4f}")
print(f"Cohen's Kappa: {kappa:.4f}")


print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
target_names = ["Entailment (VRAI)", "Neutral (À_VÉRIFIER)", "Contradiction (FAUX)"]
print(classification_report(y_test, y_pred, target_names=target_names))

Test Accuracy: 0.5378
Cohen's Kappa: 0.3067

CLASSIFICATION REPORT
                      precision    recall  f1-score   support

   Entailment (VRAI)       0.53      0.56      0.54     26180
Neutral (À_VÉRIFIER)       0.52      0.52      0.52     26180
Contradiction (FAUX)       0.57      0.54      0.55     26181

            accuracy                           0.54     78541
           macro avg       0.54      0.54      0.54     78541
        weighted avg       0.54      0.54      0.54     78541



In [11]:
# # Create a 'models' directory if it doesn't exist

# models_dir = 'models'
# if not os.path.exists(models_dir):
#     os.makedirs(models_dir)
#     print(f"Created directory: {models_dir}")


# # Save the Random Forest model
# rf_model_filename = os.path.join(models_dir, 'random_forest_model.joblib')
# joblib.dump(rf_model, rf_model_filename)
# print(f"Random Forest model saved to {rf_model_filename}")


# # Save the LinearSVC model
# svm_model_filename = os.path.join(models_dir, 'linear_svc_model.joblib')
# joblib.dump(svm_model, svm_model_filename)
# print(f"LinearSVC model saved to {svm_model_filename}")


# # Optionally save SVD if it was used
# if use_svd:
#     svd_filename = os.path.join(models_dir, 'svd_transformer.joblib')
#     joblib.dump(svd, svd_filename)
#     print(f"TruncatedSVD transformer saved to {svd_filename}")
# else:
#     print("SVD transformer not saved (use_svd=False)")

In [12]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

for i, label in enumerate(["Entailment", "Neutral", "Contradiction"]):
    class_acc = cm[i, i] / cm[i].sum()
    print(f"{label}: {class_acc:.4f}")

Confusion Matrix:
[[14634  6462  5084]
 [ 6898 13565  5717]
 [ 6087  6056 14038]]
Entailment: 0.5590
Neutral: 0.5181
Contradiction: 0.5362


In [14]:
from scipy.sparse import hstack as sp_hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.preprocessing import MaxAbsScaler
import pandas as pd

# Ensure dense arrays are sparse
feature_groups_seq = [
    ("s1_tfidf", X1_word),
    ("s2_tfidf", X2_word),
    ("word_diff", X_diff),
    ("word_mult", X_mult),
    ("cosine_word", csr_matrix(cosine_sim_word)),
    ("jaccard", csr_matrix(enhanced_feats_scaled['jaccard'])),
    ("len1", csr_matrix(enhanced_feats_scaled['len1'])),
    ("len2", csr_matrix(enhanced_feats_scaled['len2'])),
    ("len_ratio", csr_matrix(enhanced_feats_scaled['len_ratio'])),
    ("len_diff", csr_matrix(enhanced_feats_scaled['len_diff'])),
    ("neg1", csr_matrix(enhanced_feats_scaled['neg1'])),
    ("neg2", csr_matrix(enhanced_feats_scaled['neg2'])),
    ("neg_diff", csr_matrix(enhanced_feats_scaled['neg_diff'])),
]

# Helper to stack
stack = lambda groups: sp_hstack([m for _, m in groups])

# Baseline with ALL features
X_all_seq = stack(feature_groups_seq)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_all_seq, labels, test_size=0.2, random_state=42, stratify=labels
)

# Train baseline models
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=60,
    min_samples_split=20,
    min_samples_leaf=6,
    max_features='sqrt',
    class_weight='balanced',
    bootstrap=True,
    random_state=43,
    n_jobs=-1,
    verbose=0
)
rf.fit(X_tr, y_tr)
base_rf_acc = accuracy_score(y_te, rf.predict(X_te))

scaler = MaxAbsScaler()
X_tr_svm = scaler.fit_transform(X_tr)
X_te_svm = scaler.transform(X_te)
svm = LinearSVC(
    C=3,
    max_iter=5000,
    random_state=42,
    dual=False,
    class_weight='balanced',
    loss='squared_hinge'
)
svm.fit(X_tr_svm, y_tr)
base_svm_acc = accuracy_score(y_te, svm.predict(X_te_svm))

print(f"Baseline RF: {base_rf_acc:.4f} | SVM: {base_svm_acc:.4f}")

# Sequentially remove from right to left, keeping first two features
current_groups = feature_groups_seq.copy()
prev_rf_acc, prev_svm_acc = base_rf_acc, base_svm_acc
seq_results = []

# indices from last down to index 2 (keep indices 0 and 1)
for remove_idx in range(len(current_groups) - 1, 1, -1):
    removed_name, _ = current_groups[remove_idx]
    # Remove last feature
    current_groups = current_groups[:remove_idx]
    X_cur = stack(current_groups)
    X_tr_cur, X_te_cur, y_tr_cur, y_te_cur = train_test_split(
        X_cur, labels, test_size=0.2, random_state=42, stratify=labels
    )
    
    # Train RF
    rf_cur = RandomForestClassifier(
        n_estimators=400,
        max_depth=60,
        min_samples_split=20,
        min_samples_leaf=6,
        max_features='sqrt',
        class_weight='balanced',
        bootstrap=True,
        random_state=43,
        n_jobs=-1,
        verbose=0
    )
    rf_cur.fit(X_tr_cur, y_tr_cur)
    cur_rf_acc = accuracy_score(y_te_cur, rf_cur.predict(X_te_cur))
    
    # Train SVM
    scaler_cur = MaxAbsScaler()
    X_tr_svm_cur = scaler_cur.fit_transform(X_tr_cur)
    X_te_svm_cur = scaler_cur.transform(X_te_cur)
    svm_cur = LinearSVC(
        C=3,
        max_iter=5000,
        random_state=42,
        dual=False,
        class_weight='balanced',
        loss='squared_hinge'
    )
    svm_cur.fit(X_tr_svm_cur, y_tr_cur)
    cur_svm_acc = accuracy_score(y_te_cur, svm_cur.predict(X_te_svm_cur))
    
    seq_results.append({
        'removed_feature': removed_name,
        'rf_acc': cur_rf_acc,
        'svm_acc': cur_svm_acc,
        'rf_delta': prev_rf_acc - cur_rf_acc,
        'svm_delta': prev_svm_acc - cur_svm_acc
    })
    print(f"Removed {removed_name:>12} -> RF Δ {prev_rf_acc - cur_rf_acc:.4f} | SVM Δ {prev_svm_acc - cur_svm_acc:.4f}")
    prev_rf_acc, prev_svm_acc = cur_rf_acc, cur_svm_acc

seq_df = pd.DataFrame(seq_results)
print("\nSequential removal summary (larger Δ = more contribution):")
print(seq_df[['removed_feature', 'svm_delta', 'rf_delta']].to_string(index=False))

seq_ablation_results = seq_df.copy()
print("\nTip: seq_ablation_results contains per-step deltas for plotting.")

Baseline RF: 0.5778 | SVM: 0.5395
Removed     neg_diff -> RF Δ -0.0004 | SVM Δ 0.0022
Removed         neg2 -> RF Δ -0.0009 | SVM Δ 0.0014


KeyboardInterrupt: 

## Feature Ablation Analysis & Recommendations

### Features to Drop (Harmful or Negligible)

**jaccard**: SVM −0.0015, RF −0.001917 — harmful; removing improves both. Drop.

**len_diff**: SVM −0.000083, RF −0.003667 — harmful/negligible; removing helps RF. Drop.

**len1**: SVM 0.0000, RF 0.000583 — negligible; safe to drop if you want fewer features.

**neg2**: SVM 0.000583, RF −0.000167 — negligible/mixed; optional drop.

### Features to Keep (Clear Contribution)

**word_diff**: SVM 0.0020, RF 0.031417 — strong contributor, especially for Random Forest.

**word_mult**: SVM 0.025667, RF 0.002333 — strong, particularly valuable for SVM.

**cosine_word**: SVM 0.008833, RF 0.017667 — solid contribution to both models.

**len2, len_ratio, neg1**: Small positive deltas; keep if you want to maximize every bit of performance lift.


In [ ]:
import os
from datetime import datetime

out_dir = os.path.join(os.path.dirname(__file__) if '__file__' in globals() else '.', 'results')
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

saved = []

def save_df(df, filename):
    path = os.path.join(out_dir, filename)
    df.to_csv(path, index=False)
    saved.append(path)

try:
    if 'seq_ablation_results' in globals() and isinstance(seq_ablation_results, pd.DataFrame):
        save_df(seq_ablation_results, 'ablation_sequential_backward.csv')
except Exception as e:
    print(f"Skip seq_ablation_results export: {e}")

if saved:
    print("Saved CSV files:")
    for p in saved:
        print(f"- {p}")
else:
    print("No analysis DataFrames found to export.")

Saved CSV files:
- .\results\ablation_sequential_backward.csv
